# Silver: Sector-Aware Skill Extraction

Extracts technical and soft skills from job descriptions using **sector-specific skill dictionaries**.

## Sector Intelligence

- Joins with `semantic.sem_sector_map` for job sector classification
- Applies **universal skills** (programming, cloud, DevOps) to all jobs
- Applies **sector-specific skills** based on NAICS codes:
  - **Healthcare (62)**: EPIC, Cerner, HL7, HIPAA
  - **Finance (52)**: Bloomberg Terminal, Risk Modeling, SOX
  - **Manufacturing (31-33)**: CAD, PLC, Six Sigma, SCADA
  - **Tech (51)**: Microservices, API Design, SRE
  - **Professional Services (54)**: PMP, Business Analysis
  - **Retail (44-45)**: POS, Inventory, E-commerce
  - **Construction (23)**: BIM, Primavera, OSHA

## Extraction Methods

1. **Keyword matching** (deterministic, high confidence)
2. **Pattern matching** (regex patterns for skills)
3. **NLP extraction** (spaCy entity recognition)
4. **Future**: LLM-based extraction for complex cases

## Architecture

**Input**: `silver.silver_jobs_current`  
**Output**: `silver.silver_skill_mapping`  
**Mode**: Incremental (process new batches only)

## Batch Processing

- Tracks processed batches in `metadata.skill_extraction_batches`
- Automatically processes all unprocessed batches
- Idempotent: safe to re-run
- Skills normalized to taxonomy (future enhancement)

In [0]:
dbutils.widgets.text("batch_id", "", "Batch ID (leave empty for all unprocessed)")
dbutils.widgets.dropdown("extraction_method", "keyword", ["keyword", "pattern", "nlp"], "Extraction Method")

batch_id = dbutils.widgets.get("batch_id").strip()
extraction_method = dbutils.widgets.get("extraction_method")

In [0]:
import json
import re
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, ArrayType, DoubleType

CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.silver"
METADATA_SCHEMA = f"{CATALOG}.metadata"

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp_py = datetime.now()  # Python timestamp for metadata records
run_timestamp = F.current_timestamp()  # Spark timestamp for DataFrame operations

# Universal skills (apply to all sectors)
UNIVERSAL_SKILLS = {
    # Programming Languages
    "python": ["python", "python3", "py"],
    "javascript": ["javascript", "js", "node.js", "nodejs"],
    "java": ["java", "java se", "java ee"],
    "typescript": ["typescript", "ts"],
    "sql": ["sql", "t-sql", "pl/sql", "mysql", "postgresql"],
    "r": [" r ", "r programming"],
    "scala": ["scala"],
    "go": ["golang", " go "],
    "rust": ["rust"],
    "php": ["php"],
    "ruby": ["ruby", "ruby on rails"],
    "c++": ["c++", "cpp"],
    "c#": ["c#", "csharp"],
    
    # Frameworks & Libraries
    "react": ["react", "react.js", "reactjs"],
    "angular": ["angular", "angularjs"],
    "vue": ["vue", "vue.js"],
    "django": ["django"],
    "flask": ["flask"],
    "spring": ["spring", "spring boot"],
    "laravel": ["laravel"],
    "express": ["express", "express.js"],
    
    # Data & ML
    "spark": ["spark", "apache spark", "pyspark"],
    "hadoop": ["hadoop"],
    "pandas": ["pandas"],
    "numpy": ["numpy"],
    "tensorflow": ["tensorflow", "tf"],
    "pytorch": ["pytorch"],
    "scikit-learn": ["scikit-learn", "sklearn"],
    "keras": ["keras"],
    
    # Cloud & DevOps
    "aws": ["aws", "amazon web services"],
    "azure": ["azure", "microsoft azure"],
    "gcp": ["gcp", "google cloud"],
    "docker": ["docker", "containerization"],
    "kubernetes": ["kubernetes", "k8s"],
    "jenkins": ["jenkins"],
    "terraform": ["terraform"],
    "ansible": ["ansible"],
    "ci/cd": ["ci/cd", "continuous integration", "continuous deployment"],
    
    # Databases
    "mongodb": ["mongodb", "mongo"],
    "redis": ["redis"],
    "elasticsearch": ["elasticsearch", "elastic"],
    "cassandra": ["cassandra"],
    "dynamodb": ["dynamodb"],
    
    # Methodologies
    "agile": ["agile", "scrum", "kanban"],
    "devops": ["devops"],
    "machine learning": ["machine learning", "ml"],
    "data science": ["data science"],
    "ai": ["artificial intelligence", "ai"]
}

# Sector-specific skill dictionaries (NAICS-aligned)
SECTOR_SKILLS = {
    # Healthcare (NAICS 62)
    "62": {
        "epic": ["epic", "epic emr", "epic ehr"],
        "cerner": ["cerner", "cerner millennium"],
        "meditech": ["meditech"],
        "hl7": ["hl7", "hl7 fhir", "fhir"],
        "hipaa": ["hipaa", "hipaa compliance"],
        "clinical-workflows": ["clinical workflows", "patient care"],
        "medical-coding": ["icd-10", "cpt", "medical coding"],
        "telemedicine": ["telemedicine", "telehealth"],
        "radiology-pacs": ["pacs", "dicom", "radiology"]
    },
    
    # Finance & Insurance (NAICS 52)
    "52": {
        "bloomberg-terminal": ["bloomberg terminal", "bloomberg"],
        "risk-modeling": ["risk modeling", "var", "value at risk"],
        "sox-compliance": ["sox", "sarbanes-oxley"],
        "trading-systems": ["trading systems", "algo trading", "hft"],
        "regulatory-reporting": ["regulatory reporting", "finra", "sec reporting"],
        "credit-analysis": ["credit analysis", "credit risk"],
        "actuarial": ["actuarial", "actuarial science"],
        "kyc-aml": ["kyc", "aml", "know your customer", "anti-money laundering"]
    },
    
    # Manufacturing (NAICS 31-33)
    "31": {
        "cad": ["autocad", "solidworks", "catia", "cad"],
        "plc-programming": ["plc", "ladder logic", "siemens", "allen bradley"],
        "six-sigma": ["six sigma", "lean manufacturing", "kaizen"],
        "scada": ["scada", "hmi"],
        "erp-manufacturing": ["sap pp", "oracle manufacturing"],
        "quality-management": ["iso 9001", "statistical process control", "spc"],
        "cnc": ["cnc", "cnc programming", "machining"]
    },
    
    # Information/Tech (NAICS 51) - inherits universal + adds specific
    "51": {
        "cloud-architecture": ["cloud architecture", "solution architecture"],
        "microservices": ["microservices", "service mesh"],
        "api-design": ["rest api", "graphql", "api gateway"],
        "site-reliability": ["sre", "site reliability engineering"],
        "distributed-systems": ["distributed systems", "consensus algorithms"]
    },
    
    # Professional Services (NAICS 54)
    "54": {
        "consulting": ["management consulting", "strategy consulting"],
        "project-management": ["pmp", "project management", "program management"],
        "change-management": ["change management", "organizational change"],
        "business-analysis": ["business analysis", "requirements gathering"],
        "stakeholder-management": ["stakeholder management", "client management"]
    },
    
    # Retail Trade (NAICS 44-45)
    "44": {
        "pos-systems": ["pos", "point of sale", "retail systems"],
        "inventory-management": ["inventory management", "supply chain"],
        "merchandising": ["merchandising", "category management"],
        "e-commerce": ["e-commerce", "shopify", "magento"],
        "omnichannel": ["omnichannel", "omni-channel retail"]
    },
    
    # Construction (NAICS 23)
    "23": {
        "bim": ["bim", "building information modeling", "revit"],
        "project-scheduling": ["primavera", "ms project", "construction scheduling"],
        "estimating": ["cost estimating", "construction estimating"],
        "osha": ["osha", "safety compliance"],
        "construction-management": ["construction management", "general contracting"]
    }
}

# Build combined skill dictionary per sector
def get_skills_for_sector(sector_code):
    """Get combined universal + sector-specific skills"""
    skills = UNIVERSAL_SKILLS.copy()
    
    # Add sector-specific skills if available
    if sector_code and sector_code in SECTOR_SKILLS:
        skills.update(SECTOR_SKILLS[sector_code])
    
    # Also check parent codes (e.g., "31" also applies to "32", "33")
    if sector_code and len(sector_code) == 2:
        parent_code = sector_code[0]
        for code, sector_dict in SECTOR_SKILLS.items():
            if code.startswith(parent_code) and code != sector_code:
                skills.update(sector_dict)
    
    return skills

print(f"Loaded {len(UNIVERSAL_SKILLS)} universal skills + {len(SECTOR_SKILLS)} sector-specific dictionaries")

In [0]:
# Create metadata table to track skill-extracted batches
metadata_table = f"{METADATA_SCHEMA}.skill_extraction_batches"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {metadata_table} (
  batch_id STRING,
  source_name STRING,
  jobs_processed INT,
  skills_extracted INT,
  processed_at TIMESTAMP,
  run_id STRING,
  status STRING
)
USING DELTA
COMMENT 'Tracks which batches have been skill-extracted'
""")

# Define metadata schema for recording batch processing
metadata_schema = StructType([
    StructField("batch_id", StringType(), True),
    StructField("source_name", StringType(), True),
    StructField("jobs_processed", IntegerType(), True),
    StructField("skills_extracted", IntegerType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("run_id", StringType(), True),
    StructField("status", StringType(), True)
])

In [0]:
# Identify unprocessed batches
if batch_id:
    # Process specific batch if provided
    batches_to_process = [(batch_id, None)]  # (batch_id, source_name) tuple
else:
    # Find all batches in current table
    all_batches_query = f"""
        SELECT DISTINCT current_batch_id as batch_id, source_name
        FROM {SILVER_SCHEMA}.silver_jobs_current
        WHERE is_active = true AND soft_delete_flag = false
        AND current_batch_id IS NOT NULL
        AND current_batch_id != ''
        AND source_name IS NOT NULL
        AND description_raw IS NOT NULL
    """
    
    all_batches_df = spark.sql(all_batches_query)
    
    # Find already processed batches
    processed_batches_df = spark.sql(f"""
        SELECT DISTINCT batch_id, source_name
        FROM {metadata_table}
        WHERE status = 'success'
    """)
    
    # Find unprocessed batches (in current but not in metadata)
    unprocessed_df = all_batches_df.join(
        processed_batches_df,
        on=["batch_id", "source_name"],
        how="left_anti"
    ).orderBy("batch_id", "source_name")
    
    batches_to_process = [(row.batch_id, row.source_name) for row in unprocessed_df.collect()]
    
    if not batches_to_process:
        dbutils.notebook.exit({"status": "success", "message": "No unprocessed batches", "total_skills_extracted": 0})
    
    print(f"Found {len(batches_to_process)} unprocessed batch(es) to process")

In [0]:
def extract_skills_keyword(text, title="", sector_code="UNKNOWN"):
    """Extract skills using sector-aware keyword matching"""
    if not text:
        return []
    
    text_lower = (text + " " + title).lower()
    found_skills = []
    
    # Get sector-specific + universal skills
    skill_dict = get_skills_for_sector(sector_code)
    
    for skill_name, keywords in skill_dict.items():
        for keyword in keywords:
            if keyword in text_lower:
                # Find the actual occurrence for evidence
                match = re.search(r'.{0,30}' + re.escape(keyword) + r'.{0,30}', text_lower, re.IGNORECASE)
                evidence = match.group(0) if match else keyword
                
                found_skills.append({
                    "skill_name_raw": keyword,
                    "skill_name_normalized": skill_name,
                    "extraction_method": "KEYWORD",
                    "evidence_text": evidence,
                    "confidence": 0.95,  # High confidence for exact keyword match
                    "sector_context": sector_code if sector_code != "UNKNOWN" else None
                })
                break  # Only match once per skill
    
    return found_skills

# Register as UDF
skill_extract_udf = F.udf(extract_skills_keyword, ArrayType(StructType([
    StructField("skill_name_raw", StringType()),
    StructField("skill_name_normalized", StringType()),
    StructField("extraction_method", StringType()),
    StructField("evidence_text", StringType()),
    StructField("confidence", DoubleType()),
    StructField("sector_context", StringType())
])))

# Function registered

In [0]:
# Initialize tracking
total_jobs_processed = 0
total_skills_extracted = 0
processed_batch_count = 0
failed_batches = []

# Ensure skill_mapping table exists
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.silver_skill_mapping (
  skill_mapping_id STRING,
  enterprise_job_id STRING,
  skill_name_raw STRING,
  skill_name_normalized STRING,
  extraction_method STRING,
  evidence_text STRING,
  confidence DECIMAL(5,4),
  batch_id STRING,
  extracted_at TIMESTAMP
)
USING DELTA
COMMENT 'Sector-aware skill extraction mapping'
""")

# Add sector columns if they don't exist (schema evolution)
try:
    current_schema = spark.table(f"{SILVER_SCHEMA}.silver_skill_mapping").schema
    current_columns = [field.name for field in current_schema.fields]
    
    missing_columns = []
    if "sector_code" not in current_columns:
        missing_columns.append("ADD COLUMN sector_code STRING")
    if "sector_name" not in current_columns:
        missing_columns.append("ADD COLUMN sector_name STRING")
    if "sector_context" not in current_columns:
        missing_columns.append("ADD COLUMN sector_context STRING COMMENT 'Sector that triggered this skill extraction'")
    
    if missing_columns:
        for col_stmt in missing_columns:
            spark.sql(f"ALTER TABLE {SILVER_SCHEMA}.silver_skill_mapping {col_stmt}")
        print(f"Added {len(missing_columns)} sector columns to skill_mapping table")
except Exception as e:
    print(f"Schema check: {str(e)}")

# Process each batch
for current_batch_id, current_source in batches_to_process:
    try:
        print(f"Extracting skills from batch {current_batch_id} ({current_source})...", end=" ")
        
        # Load jobs for this batch with sector information
        batch_query = f"""
            SELECT 
                j.enterprise_job_id,
                j.source_name,
                j.title_raw,
                j.description_raw,
                COALESCE(s.canonical_sector_code, 'UNKNOWN') as sector_code,
                s.canonical_sector_name as sector_name
            FROM {SILVER_SCHEMA}.silver_jobs_current j
            LEFT JOIN workspace.semantic.sem_sector_map s
                ON j.enterprise_job_id = s.enterprise_job_id
            WHERE j.current_batch_id = '{current_batch_id}'
            AND j.is_active = true AND j.soft_delete_flag = false
            AND j.description_raw IS NOT NULL
        """
        
        if current_source:
            batch_query += f" AND source_name = '{current_source}'"
        
        jobs_df = spark.sql(batch_query)
        job_count = jobs_df.count()
        
        if job_count == 0:
            print("No jobs found")
            continue
        
        # Apply sector-aware skill extraction
        skills_extracted_df = jobs_df.withColumn(
            "extracted_skills",
            skill_extract_udf(F.col("description_raw"), F.col("title_raw"), F.col("sector_code"))
        )
        
        # Explode skills array to individual rows
        skills_exploded_df = skills_extracted_df.select(
            F.col("enterprise_job_id"),
            F.col("source_name"),
            F.col("sector_code"),
            F.col("sector_name"),
            F.explode("extracted_skills").alias("skill")
        ).select(
            F.col("enterprise_job_id"),
            F.col("source_name"),
            F.col("sector_code"),
            F.col("sector_name"),
            F.col("skill.skill_name_raw"),
            F.col("skill.skill_name_normalized"),
            F.col("skill.extraction_method"),
            F.col("skill.evidence_text"),
            F.col("skill.confidence"),
            F.col("skill.sector_context")
        )
        
        # Add metadata
        skills_final_df = skills_exploded_df.withColumn(
            "skill_mapping_id", F.expr("uuid()")
        ).withColumn(
            "batch_id", F.lit(current_batch_id)
        ).withColumn(
            "extracted_at", run_timestamp
        )
        
        skills_count = skills_final_df.count()
        
        # Write extracted skills
        if skills_count > 0:
            skills_final_df.select(
                "skill_mapping_id",
                "enterprise_job_id",
                "skill_name_raw",
                "skill_name_normalized",
                "extraction_method",
                "evidence_text",
                F.col("confidence").cast("decimal(5,4)"),
                "sector_code",
                "sector_name",
                "sector_context",
                "batch_id",
                "extracted_at"
            ).write.format("delta").mode("append").saveAsTable(f"{SILVER_SCHEMA}.silver_skill_mapping")
        
        # Record batch processing in metadata
        safe_source = current_source if current_source else "unknown"
        metadata_record = spark.createDataFrame([
            (current_batch_id, safe_source, int(job_count), int(skills_count), run_timestamp_py, run_id, "success")
        ], schema=metadata_schema)
        
        metadata_record.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(metadata_table)
        
        # Update tracking
        total_jobs_processed += job_count
        total_skills_extracted += skills_count
        processed_batch_count += 1
        
        print(f"✓ Jobs:{job_count} Skills:{skills_count}")
        
    except Exception as e:
        print(f"✗ {str(e)}")
        failed_batches.append((current_batch_id, current_source, str(e)))
        
        # Record failure in metadata
        try:
            safe_source = current_source if current_source else "unknown"
            failure_record = spark.createDataFrame([
                (current_batch_id, safe_source, 0, 0, run_timestamp_py, run_id, f"failed: {str(e)}")
            ], schema=metadata_schema)
            
            failure_record.write \
                .format("delta") \
                .mode("append") \
                .saveAsTable(metadata_table)
        except:
            pass
        
        continue

print(f"\nProcessed {processed_batch_count} batches, {total_jobs_processed} jobs, extracted {total_skills_extracted} skills")
if failed_batches:
    print(f"Failed: {len(failed_batches)} batches")

In [0]:
# Show extraction history
if processed_batch_count > 0:
    metadata_df = spark.sql(f"""
        SELECT batch_id, source_name, jobs_processed, skills_extracted, processed_at, status
        FROM {metadata_table}
        ORDER BY processed_at DESC
        LIMIT 20
    """)
    display(metadata_df)
    
    # Show sample of extracted skills with sector context
    sample_df = spark.sql(f"""
        SELECT skill_name_normalized, sector_code, sector_name, evidence_text, confidence, sector_context
        FROM {SILVER_SCHEMA}.silver_skill_mapping
        ORDER BY extracted_at DESC
        LIMIT 10
    """)
    display(sample_df)
    
    # Show skills by sector breakdown
    sector_breakdown_df = spark.sql(f"""
        SELECT 
            sector_code,
            sector_name,
            COUNT(DISTINCT enterprise_job_id) as jobs_count,
            COUNT(*) as skills_extracted,
            ROUND(AVG(confidence), 3) as avg_confidence
        FROM {SILVER_SCHEMA}.silver_skill_mapping
        WHERE batch_id IN (
            SELECT batch_id FROM {metadata_table}
            WHERE run_id = '{run_id}'
        )
        GROUP BY sector_code, sector_name
        ORDER BY skills_extracted DESC
    """)
    display(sector_breakdown_df)

# Exit summary
result = {
    "status": "success" if len(failed_batches) == 0 else "partial_success",
    "run_id": run_id,
    "batches_processed": processed_batch_count,
    "batches_failed": len(failed_batches),
    "jobs_processed": total_jobs_processed,
    "skills_extracted": total_skills_extracted,
    "avg_skills_per_job": round(total_skills_extracted / total_jobs_processed, 2) if total_jobs_processed > 0 else 0,
    "metadata_table": metadata_table
}

dbutils.notebook.exit(json.dumps(result))